# AWFNO-H10-K5 for 1D Burgers

This notebook runs the missing Burgers comparison using the paper's AWFNO
spatial architecture and K=5 output window, while giving the model the first
H=10 exact states to match the UNO-H10 information protocol.

Protocol:

- data: `1D_Burgers_Sols_Nu0.001.hdf5`;
- nested training sets N = 25, 50, 100, 200, 400, 800;
- validation/test sets: 50/100 fixed trajectories;
- input: 10 consecutive states;
- output: 5 future states per call;
- inference: the first 10 states are exact and every later state is recursive;
- architecture: width 32, 4 AFF layers, 16 Fourier modes, radius 3;
- optimization: 200 epochs, AdamW, lr 1e-3, weight decay 1e-6;
- metrics: absolute physical full-trajectory and final-time L2.

This is **AWFNO-H10-K5**, not the original paper's u0-only AWFNO. Keep it in
the same H10 protocol table as UNO-H10.



In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')



In [ ]:
import os, gc, csv, json, time, random, hashlib, signal
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader



## Configuration



In [ ]:
CONFIG = {
    "DATA_PATH": "/content/gdrive/MyDrive/anus/1D_Burgers_Sols_Nu0.001.hdf5",
    "OUTPUT_ROOT": "/content/gdrive/MyDrive/anus/burgers_awfno_h10_k5_paper_v1",

    "TRAIN_POOL_SIZE": 800,
    "N_VAL": 50,
    "N_TEST": 100,
    "SAMPLE_SIZES": [25, 50, 100, 200, 400, 800],
    "RUN_SAMPLE_SIZES": [25, 50, 100, 200, 400, 800],
    "TARGET_NX": 512,
    "TIME_SUBSAMPLE": 1,

    "HISTORY_LENGTH": 10,
    "WINDOW_K": 5,
    "WINDOW_STRIDE": 5,

    "WIDTH": 32,
    "LAYERS": 4,
    "MODES": 16,
    "LOCAL_RADIUS": 3,

    "EPOCHS": 200,
    "BATCH_SIZE": 128,
    "LR": 1.0e-3,
    "WEIGHT_DECAY": 1.0e-6,
    "FINAL_LOSS_WEIGHT": 1.0,
    "GRAD_CLIP": 1.0,

    "SEED": 1234,
    "FORCE_RETRAIN": False,
    "CHECKPOINT_EVERY": 10,
    "INFERENCE_REPEATS": 3,
    "NUM_WORKERS": 0,

    "SAVE_PLOTS": True,
    "SAVE_ROLLOUTS": False,
    "USE_TRAPEZOIDAL_TIME_L2": True,

    # Set True only in the last cell after confirming all results were saved.
    "STOP_RUNTIME_AT_END": True,
}

assert CONFIG["HISTORY_LENGTH"] == 10
assert CONFIG["WINDOW_K"] == 5
assert CONFIG["WIDTH"] == 32
assert CONFIG["LAYERS"] == 4
assert CONFIG["MODES"] == 16
assert CONFIG["LOCAL_RADIUS"] == 3

DATA_PATH = Path(CONFIG["DATA_PATH"])
OUTPUT_ROOT = Path(CONFIG["OUTPUT_ROOT"])
TAG = (
    f"H{CONFIG['HISTORY_LENGTH']}_K{CONFIG['WINDOW_K']}_"
    f"W{CONFIG['WIDTH']}_L{CONFIG['LAYERS']}_M{CONFIG['MODES']}_"
    f"R{CONFIG['LOCAL_RADIUS']}_E{CONFIG['EPOCHS']}"
)
RUN_ROOT = OUTPUT_ROOT / TAG
MODEL_DIR = RUN_ROOT / "models"
RESULT_DIR = RUN_ROOT / "results"
PLOT_DIR = RUN_ROOT / "plots"
DATA_DIR = RUN_ROOT / "data"
for directory in (MODEL_DIR, RESULT_DIR, PLOT_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Data   :", DATA_PATH)
print("Exists :", DATA_PATH.exists())
print("Output :", RUN_ROOT)



## Reproducibility and device



In [ ]:
SEED = int(CONFIG["SEED"])
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU   :", torch.cuda.get_device_name(0))



## HDF5 loading and fixed split



In [ ]:
def list_datasets(handle):
    found = []
    handle.visititems(lambda name, obj: found.append((name, tuple(obj.shape), obj.dtype)) if isinstance(obj, h5py.Dataset) else None)
    return found


def choose_solution_key(handle):
    candidates = []
    for name, shape, dtype in list_datasets(handle):
        squeezed = tuple(d for d in shape if d != 1)
        if len(squeezed) == 3 and np.issubdtype(dtype, np.number):
            candidates.append((int(np.prod(shape)), name))
    if not candidates:
        raise ValueError("Could not find a three-dimensional solution dataset in the HDF5 file.")
    return max(candidates)[1]


def choose_coordinate_key(handle, kind):
    preferred = {
        "x": ["x-coordinate", "x_coordinate", "x", "grid/x", "coords/x"],
        "t": ["t-coordinate", "t_coordinate", "t", "time", "grid/t", "coords/t"],
    }[kind]
    names = [name for name, shape, _ in list_datasets(handle) if len(shape) == 1]
    lowered = {name.lower(): name for name in names}
    for item in preferred:
        if item.lower() in lowered:
            return lowered[item.lower()]
    for name in names:
        leaf = name.lower().split("/")[-1]
        if kind == "x" and (leaf == "x" or "x-coordinate" in leaf or "x_coordinate" in leaf):
            return name
        if kind == "t" and (leaf in {"t", "time"} or "t-coordinate" in leaf or "t_coordinate" in leaf):
            return name
    return None


def to_nxt(raw, x=None, t=None):
    array = np.asarray(raw, dtype=np.float32)
    while array.ndim == 4 and 1 in array.shape:
        array = np.squeeze(array, axis=list(array.shape).index(1))
    if array.ndim != 3:
        raise ValueError(f"Expected solution shape (N,Nx,Nt) or (N,Nt,Nx), got {array.shape}.")
    spatial, temporal = None, None
    if x is not None:
        axes = [axis for axis in (1, 2) if array.shape[axis] == len(x)]
        if len(axes) == 1:
            spatial = axes[0]
    if t is not None:
        axes = [axis for axis in (1, 2) if array.shape[axis] == len(t)]
        if len(axes) == 1:
            temporal = axes[0]
    if spatial is None and temporal is not None:
        spatial = 3 - temporal
    if temporal is None and spatial is not None:
        temporal = 3 - spatial
    if spatial is None or temporal is None or spatial == temporal:
        # Burgers has many more spatial points than saved time snapshots.
        spatial = 1 if array.shape[1] >= array.shape[2] else 2
        temporal = 3 - spatial
    if (spatial, temporal) == (1, 2):
        return array.astype(np.float32, copy=False)
    return np.transpose(array, (0, 2, 1)).astype(np.float32, copy=False)


def load_burgers(path):
    needed = CONFIG["TRAIN_POOL_SIZE"] + CONFIG["N_VAL"] + CONFIG["N_TEST"]
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")
    with h5py.File(path, "r") as handle:
        print("HDF5 datasets:")
        for item in list_datasets(handle):
            print(" ", item)
        solution_key = choose_solution_key(handle)
        x_key = choose_coordinate_key(handle, "x")
        t_key = choose_coordinate_key(handle, "t")
        dataset = handle[solution_key]
        if dataset.shape[0] < needed:
            raise ValueError(f"Need {needed} trajectories, but {solution_key} contains {dataset.shape[0]}.")
        raw = np.asarray(dataset[:needed], dtype=np.float32)
        x = None if x_key is None else np.asarray(handle[x_key], dtype=np.float32).reshape(-1)
        t = None if t_key is None else np.asarray(handle[t_key], dtype=np.float32).reshape(-1)
        print("Selected solution/x/t:", solution_key, x_key, t_key)
    U = to_nxt(raw, x=x, t=t)
    if x is None or len(x) != U.shape[1]:
        x = np.linspace(0.0, 1.0, U.shape[1], endpoint=False, dtype=np.float32)
    if t is None or len(t) != U.shape[2]:
        t = np.arange(U.shape[2], dtype=np.float32)
    if CONFIG["TIME_SUBSAMPLE"] > 1:
        step = int(CONFIG["TIME_SUBSAMPLE"])
        U, t = U[:, :, ::step], t[::step]
    target_nx = int(CONFIG["TARGET_NX"])
    if U.shape[1] != target_nx:
        if U.shape[1] % target_nx != 0:
            raise ValueError(f"Cannot stride-downsample Nx={U.shape[1]} to {target_nx}.")
        stride = U.shape[1] // target_nx
        U, x = U[:, ::stride, :], x[::stride]
    if not np.all(np.isfinite(U)):
        raise FloatingPointError("Dataset contains NaN or Inf.")
    return U.astype(np.float32), x.astype(np.float32), t.astype(np.float32)


def trajectory_hashes(U):
    return [hashlib.sha1(np.ascontiguousarray(item, dtype=np.float32).tobytes()).hexdigest() for item in U]


def assert_no_overlap(A, B, name_a, name_b):
    overlap = set(trajectory_hashes(A)).intersection(trajectory_hashes(B))
    print(f"Exact overlap {name_a}/{name_b}: {len(overlap)}")
    if overlap:
        raise RuntimeError(f"Duplicate trajectories between {name_a} and {name_b}.")


U_all, x_grid, t_grid = load_burgers(DATA_PATH)
n_pool = int(CONFIG["TRAIN_POOL_SIZE"])
n_val = int(CONFIG["N_VAL"])
n_test = int(CONFIG["N_TEST"])
U_train_pool = U_all[:n_pool]
U_val = U_all[n_pool:n_pool + n_val]
U_test = U_all[n_pool + n_val:n_pool + n_val + n_test]
train_ids = np.arange(0, n_pool)
val_ids = np.arange(n_pool, n_pool + n_val)
test_ids = np.arange(n_pool + n_val, n_pool + n_val + n_test)

NX, NT = U_all.shape[1], U_all.shape[2]
DX = float(np.mean(np.diff(x_grid.astype(np.float64)))) if NX > 1 else 1.0
DT = float(np.mean(np.diff(t_grid.astype(np.float64)))) if NT > 1 else 1.0
print("U/train/val/test:", U_all.shape, U_train_pool.shape, U_val.shape, U_test.shape)
print("NX, NT, DX, DT  :", NX, NT, DX, DT)
assert_no_overlap(U_train_pool, U_val, "train", "validation")
assert_no_overlap(U_train_pool, U_test, "train", "test")
assert_no_overlap(U_val, U_test, "validation", "test")

split_path = DATA_DIR / "fixed_contiguous_split.npz"
np.savez_compressed(split_path, train_ids=train_ids, val_ids=val_ids, test_ids=test_ids, x=x_grid, t=t_grid)
print("Saved split:", split_path)
del U_all
gc.collect()



## Physical L2 metrics



In [ ]:
def time_weights(nt):
    weights = np.ones(nt, dtype=np.float64)
    if CONFIG["USE_TRAPEZOIDAL_TIME_L2"] and nt > 1:
        weights[0] = weights[-1] = 0.5
    return weights


def time_weights_torch(nt, device, dtype):
    weights = torch.ones(nt, device=device, dtype=dtype)
    if CONFIG["USE_TRAPEZOIDAL_TIME_L2"] and nt > 1:
        weights[0] = weights[-1] = 0.5
    return weights


def l2_per_timestep_np(prediction, truth):
    if prediction.shape != truth.shape:
        raise ValueError((prediction.shape, truth.shape))
    difference = prediction.astype(np.float64) - truth.astype(np.float64)
    return np.sqrt(np.sum(difference * difference, axis=1) * DX + 1.0e-30)


def rollout_metrics(prediction, truth):
    l2_t = l2_per_timestep_np(prediction, truth)
    full = np.sqrt(np.sum(l2_t * l2_t * time_weights(l2_t.shape[1])[None, :], axis=1) * DT + 1.0e-30)
    final = l2_t[:, -1]
    return {
        "full_mean": float(full.mean()), "full_std": float(full.std()), "full_median": float(np.median(full)),
        "final_mean": float(final.mean()), "final_std": float(final.std()), "final_median": float(np.median(final)),
        "mean_t": l2_t.mean(axis=0), "std_t": l2_t.std(axis=0), "median_t": np.median(l2_t, axis=0),
        "all_full": full, "all_final": final,
    }


def window_l2_batch(prediction, truth):
    weights = time_weights_torch(prediction.shape[-1], prediction.device, prediction.dtype).view(1, 1, -1)
    return torch.sqrt(torch.sum((prediction - truth) ** 2 * weights, dim=(1, 2)) * DX * DT + 1.0e-30)


def field_l2_batch(prediction, truth):
    return torch.sqrt(torch.sum((prediction - truth) ** 2, dim=1) * DX + 1.0e-30)



## H10-to-K5 training windows and normalization



In [ ]:
H = int(CONFIG["HISTORY_LENGTH"])
K = int(CONFIG["WINDOW_K"])


def make_window_starts(nt):
    last_start = nt - H - K
    if last_start < 0:
        raise ValueError(f"Need NT >= H+K, got {nt}.")
    starts = list(range(0, last_start + 1, int(CONFIG["WINDOW_STRIDE"])))
    # Include one terminal-aligned exact training window when the grid length is
    # not divisible by K. This exposes the model to the final snapshot.
    if starts[-1] != last_start:
        starts.append(last_start)
    return starts


def build_windows(U, starts):
    total = U.shape[0] * len(starts)
    history = np.empty((total, U.shape[1], H), dtype=np.float32)
    future = np.empty((total, U.shape[1], K), dtype=np.float32)
    q = 0
    for trajectory in U:
        for start in starts:
            history[q] = trajectory[:, start:start + H]
            future[q] = trajectory[:, start + H:start + H + K]
            q += 1
    return history, future


class WindowDataset(Dataset):
    def __init__(self, history, future):
        self.history = torch.as_tensor(history, dtype=torch.float32)
        self.future = torch.as_tensor(future, dtype=torch.float32)

    def __len__(self):
        return self.history.shape[0]

    def __getitem__(self, index):
        return {"history": self.history[index], "future": self.future[index]}


class GaussianNormalizer:
    def __init__(self, values=None, mean=None, std=None, eps=1.0e-6):
        if values is not None:
            self.mean = values.mean().detach().clone()
            self.std = values.std().detach().clone()
        else:
            self.mean = torch.tensor(float(mean), dtype=torch.float32)
            self.std = torch.tensor(float(std), dtype=torch.float32)
        self.eps = float(eps)

    def encode(self, values):
        return (values - self.mean) / (self.std + self.eps)

    def decode(self, values):
        return values * (self.std + self.eps) + self.mean

    def to(self, device):
        self.mean, self.std = self.mean.to(device), self.std.to(device)
        return self

    def state(self):
        return {"mean": float(self.mean.detach().cpu()), "std": float(self.std.detach().cpu()), "eps": self.eps}


WINDOW_STARTS = make_window_starts(NT)
val_history, val_future = build_windows(U_val, WINDOW_STARTS)
print("Training window starts:", WINDOW_STARTS)
print("Validation windows    :", val_history.shape, val_future.shape)



## Paper-style one-dimensional AFF architecture



In [ ]:
class SpectralConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, modes):
        super().__init__()
        scale = 1.0 / np.sqrt(in_channels * out_channels)
        self.weight = nn.Parameter(scale * torch.randn(in_channels, out_channels, modes, dtype=torch.cfloat))

    def forward(self, values):
        batch, _, nx = values.shape
        transformed = torch.fft.rfft(values.float(), dim=-1, norm="ortho")
        output = torch.zeros(batch, self.weight.shape[1], nx // 2 + 1, dtype=torch.cfloat, device=values.device)
        modes = min(self.weight.shape[-1], transformed.shape[-1])
        output[:, :, :modes] = torch.einsum("bim,iom->bom", transformed[:, :, :modes], self.weight[:, :, :modes])
        return torch.fft.irfft(output, n=nx, dim=-1, norm="ortho")


class AFFLayer1d(nn.Module):
    def __init__(self, width, modes, radius):
        super().__init__()
        self.local_1 = nn.Conv1d(width, width, 2 * radius + 1, padding=radius, padding_mode="circular")
        self.fourier = SpectralConv1d(width, width, modes)
        self.local_2 = nn.Conv1d(width, width, 2 * radius + 1, padding=radius, padding_mode="circular")
        self.activation = nn.GELU()

    def forward(self, values):
        first = values + self.activation(self.local_1(values))
        second = first + self.activation(self.fourier(first))
        return second + self.activation(self.local_2(second))


class AWFNOHistory1d(nn.Module):
    def __init__(self, x):
        super().__init__()
        width = int(CONFIG["WIDTH"])
        x_tensor = torch.as_tensor(x, dtype=torch.float32)
        xn = (x_tensor - x_tensor.min()) / (x_tensor.max() - x_tensor.min() + 1.0e-12)
        grid = torch.stack((xn, torch.sin(2 * np.pi * xn), torch.cos(2 * np.pi * xn)), dim=-1)
        self.register_buffer("grid", grid.unsqueeze(0), persistent=False)
        self.input_projection = nn.Linear(H + 3, width)
        self.blocks = nn.ModuleList([AFFLayer1d(width, int(CONFIG["MODES"]), int(CONFIG["LOCAL_RADIUS"])) for _ in range(int(CONFIG["LAYERS"]))])
        self.output_1 = nn.Linear(width, 128)
        self.output_2 = nn.Linear(128, K)

    def forward(self, history_normalized):
        if history_normalized.ndim != 3 or history_normalized.shape[-1] != H:
            raise ValueError(f"Expected (B,Nx,H={H}), got {tuple(history_normalized.shape)}.")
        grid = self.grid.expand(history_normalized.shape[0], -1, -1)
        values = self.input_projection(torch.cat((history_normalized, grid), dim=-1)).permute(0, 2, 1).contiguous()
        for block in self.blocks:
            values = block(values)
        values = values.permute(0, 2, 1).contiguous()
        return self.output_2(F.gelu(self.output_1(values))).contiguous()


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


def count_real_scalars(model):
    return sum(parameter.numel() * (2 if torch.is_complex(parameter) else 1) for parameter in model.parameters() if parameter.requires_grad)


smoke_model = AWFNOHistory1d(x_grid).to(DEVICE)
with torch.no_grad():
    smoke_output = smoke_model(torch.zeros(2, NX, H, device=DEVICE))
assert smoke_output.shape == (2, NX, K), smoke_output.shape
print("Smoke output      :", tuple(smoke_output.shape))
print("Trainable params  :", f"{count_parameters(smoke_model):,}")
print("Stored real scalar:", f"{count_real_scalars(smoke_model):,}")
del smoke_model, smoke_output
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()



## Training, validation, and resumable checkpoints



In [ ]:
def clean_state_dict(state):
    return {key.removeprefix("_orig_mod."): value for key, value in state.items()}


def torch_load_safe(path, map_location=None):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)


def byte_rng_state(value):
    if value is None:
        return None
    if isinstance(value, np.ndarray):
        value = torch.from_numpy(value)
    elif not torch.is_tensor(value):
        value = torch.tensor(value)
    return value.detach().cpu().to(torch.uint8).contiguous()


def make_predict(model, history_normalizer, future_normalizer):
    def predict(history):
        return future_normalizer.decode(model(history_normalizer.encode(history)))
    return predict


def normalized_window_loss(prediction, truth, normalizer):
    pred_norm, truth_norm = normalizer.encode(prediction), normalizer.encode(truth)
    reconstruction = F.mse_loss(pred_norm, truth_norm)
    final = F.mse_loss(pred_norm[:, :, -1], truth_norm[:, :, -1])
    return reconstruction + float(CONFIG["FINAL_LOSS_WEIGHT"]) * final


def train_epoch(model, loader, optimizer, predict, future_normalizer):
    model.train()
    loss_total = l2_total = count = 0.0
    for batch in loader:
        history = batch["history"].to(DEVICE, non_blocking=True)
        truth = batch["future"].to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        prediction = predict(history)
        loss = normalized_window_loss(prediction, truth, future_normalizer)
        if not torch.isfinite(loss):
            raise FloatingPointError("Non-finite AWFNO loss; do not report this checkpoint.")
        loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), float(CONFIG["GRAD_CLIP"]))
        if not torch.isfinite(grad_norm):
            raise FloatingPointError("Non-finite AWFNO gradients; do not report this checkpoint.")
        optimizer.step()
        batch_size = history.shape[0]
        loss_total += float(loss.detach()) * batch_size
        l2_total += float(window_l2_batch(prediction.detach(), truth).mean()) * batch_size
        count += batch_size
    return loss_total / count, l2_total / count


@torch.no_grad()
def eval_epoch(model, loader, predict, future_normalizer):
    model.eval()
    loss_total = l2_total = final_total = count = 0.0
    for batch in loader:
        history = batch["history"].to(DEVICE, non_blocking=True)
        truth = batch["future"].to(DEVICE, non_blocking=True)
        prediction = predict(history)
        loss = normalized_window_loss(prediction, truth, future_normalizer)
        batch_size = history.shape[0]
        loss_total += float(loss) * batch_size
        l2_total += float(window_l2_batch(prediction, truth).mean()) * batch_size
        final_total += float(field_l2_batch(prediction[:, :, -1], truth[:, :, -1]).mean()) * batch_size
        count += batch_size
    return loss_total / count, l2_total / count, final_total / count


def fit_or_load(n_train, train_history, train_future):
    run_dir = MODEL_DIR / f"N{n_train}"
    run_dir.mkdir(parents=True, exist_ok=True)
    latest_path, best_path, done_path = run_dir / "latest.pt", run_dir / "best.pt", run_dir / "done.pt"

    history_normalizer = GaussianNormalizer(torch.as_tensor(train_history)).to(DEVICE)
    future_normalizer = GaussianNormalizer(torch.as_tensor(train_future)).to(DEVICE)
    model = AWFNOHistory1d(x_grid).to(DEVICE)
    predict = make_predict(model, history_normalizer, future_normalizer)

    if done_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        saved = torch_load_safe(done_path, map_location=DEVICE)
        model.load_state_dict(clean_state_dict(saved["state_dict"]), strict=True)
        print("Loading completed checkpoint:", done_path)
        return {"model": model, "predict": predict, "history": saved["history"], "best_val": float(saved["best_val"]), "best_epoch": int(saved["best_epoch"]), "train_time_sec": float(saved["train_time_sec"]), "checkpoint": str(done_path)}

    generator = torch.Generator().manual_seed(SEED + n_train)
    train_loader = DataLoader(WindowDataset(train_history, train_future), batch_size=int(CONFIG["BATCH_SIZE"]), shuffle=True, generator=generator, num_workers=int(CONFIG["NUM_WORKERS"]), pin_memory=PIN_MEMORY)
    val_loader = DataLoader(WindowDataset(val_history, val_future), batch_size=int(CONFIG["BATCH_SIZE"]), shuffle=False, num_workers=int(CONFIG["NUM_WORKERS"]), pin_memory=PIN_MEMORY)
    optimizer = torch.optim.AdamW(model.parameters(), lr=float(CONFIG["LR"]), weight_decay=float(CONFIG["WEIGHT_DECAY"]))
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(CONFIG["EPOCHS"]))

    start_epoch, best_val, best_epoch, elapsed_before = 1, float("inf"), 0, 0.0
    history = {key: [] for key in ("epoch", "train_loss", "train_l2", "val_loss", "val_l2", "val_final_l2")}
    if latest_path.exists() and not CONFIG["FORCE_RETRAIN"]:
        latest = torch_load_safe(latest_path, map_location=DEVICE)
        model.load_state_dict(clean_state_dict(latest["state_dict"]), strict=True)
        optimizer.load_state_dict(latest["optimizer"])
        scheduler.load_state_dict(latest["scheduler"])
        start_epoch = int(latest["epoch"]) + 1
        best_val, best_epoch = float(latest["best_val"]), int(latest["best_epoch"])
        elapsed_before = float(latest.get("train_time_sec", 0.0))
        history = latest["history"]
        rng_state = byte_rng_state(latest.get("shuffle_generator_state"))
        if rng_state is not None:
            generator.set_state(rng_state)
        print(f"Resuming N={n_train} at epoch {start_epoch}.")

    started = time.time()
    for epoch in range(start_epoch, int(CONFIG["EPOCHS"]) + 1):
        train_loss, train_l2 = train_epoch(model, train_loader, optimizer, predict, future_normalizer)
        val_loss, val_l2, val_final = eval_epoch(model, val_loader, predict, future_normalizer)
        scheduler.step()
        for key, value in (("epoch", epoch), ("train_loss", train_loss), ("train_l2", train_l2), ("val_loss", val_loss), ("val_l2", val_l2), ("val_final_l2", val_final)):
            history[key].append(value)
        elapsed = elapsed_before + time.time() - started
        if val_l2 < best_val:
            best_val, best_epoch = val_l2, epoch
            torch.save({"state_dict": clean_state_dict(model.state_dict()), "best_val": best_val, "best_epoch": best_epoch}, best_path)
        if epoch == 1 or epoch % 10 == 0 or epoch == int(CONFIG["EPOCHS"]):
            print(f"[AWFNO-H10-K5] N={n_train} {epoch:04d}/{CONFIG['EPOCHS']} | train loss {train_loss:.6e} | train L2 {train_l2:.6e} | val loss {val_loss:.6e} | val L2 {val_l2:.6e} | val final {val_final:.6e}")
        if epoch % int(CONFIG["CHECKPOINT_EVERY"]) == 0 or epoch == int(CONFIG["EPOCHS"]):
            torch.save({"state_dict": clean_state_dict(model.state_dict()), "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(), "epoch": epoch, "history": history, "best_val": best_val, "best_epoch": best_epoch, "train_time_sec": elapsed, "shuffle_generator_state": generator.get_state().cpu(), "history_normalizer": history_normalizer.state(), "future_normalizer": future_normalizer.state(), "config": CONFIG}, latest_path)

    if not best_path.exists():
        raise RuntimeError("No valid best checkpoint was produced.")
    best = torch_load_safe(best_path, map_location=DEVICE)
    model.load_state_dict(clean_state_dict(best["state_dict"]), strict=True)
    train_time = elapsed_before + time.time() - started
    done = {"state_dict": clean_state_dict(model.state_dict()), "history": history, "best_val": best_val, "best_epoch": best_epoch, "train_time_sec": train_time, "history_normalizer": history_normalizer.state(), "future_normalizer": future_normalizer.state(), "config": CONFIG}
    torch.save(done, done_path)
    return {"model": model, "predict": predict, "history": history, "best_val": best_val, "best_epoch": best_epoch, "train_time_sec": train_time, "checkpoint": str(done_path)}



## Recursive H10/K5 rollout



In [ ]:
@torch.no_grad()
def rollout(model, predict, truth):
    model.eval()
    output = np.zeros_like(truth, dtype=np.float32)
    output[:, :, :H] = truth[:, :, :H]
    for first in range(0, truth.shape[0], int(CONFIG["BATCH_SIZE"])):
        last = min(first + int(CONFIG["BATCH_SIZE"]), truth.shape[0])
        history = torch.as_tensor(truth[first:last, :, :H], dtype=torch.float32, device=DEVICE)
        cursor = H
        while cursor < truth.shape[-1]:
            future = predict(history)
            use = min(K, truth.shape[-1] - cursor)
            output[first:last, :, cursor:cursor + use] = future[:, :, :use].detach().cpu().numpy()
            if cursor + use < truth.shape[-1]:
                new_values = future[:, :, :use].detach()
                history = new_values[:, :, -H:] if use >= H else torch.cat((history[:, :, use:], new_values), dim=-1)
            cursor += use
    return output


def timed_rollout(model, predict, truth):
    _ = rollout(model, predict, truth[:min(2, len(truth))])
    times, prediction = [], None
    for _ in range(int(CONFIG["INFERENCE_REPEATS"])):
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        start = time.perf_counter()
        prediction = rollout(model, predict, truth)
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    return prediction, float(np.median(times)), times


def plot_history(history, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    figure, axes = plt.subplots(1, 2, figsize=(12, 4.2))
    axes[0].semilogy(history["epoch"], history["train_loss"], label="train")
    axes[0].semilogy(history["epoch"], history["val_loss"], label="validation")
    axes[0].set(xlabel="epoch", ylabel="normalized H10-to-K5 loss")
    axes[0].legend()
    axes[1].semilogy(history["epoch"], history["train_l2"], label="train")
    axes[1].semilogy(history["epoch"], history["val_l2"], label="validation")
    axes[1].set(xlabel="epoch", ylabel="absolute window L2")
    axes[1].legend()
    figure.suptitle(f"AWFNO-H10-K5, N={n_train}")
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f"N{n_train}_training.png", dpi=150)
    plt.close(figure)


def plot_l2_time(metrics, n_train):
    if not CONFIG["SAVE_PLOTS"]:
        return
    figure, axis = plt.subplots(figsize=(7.5, 4.5))
    axis.plot(t_grid, metrics["mean_t"], label="mean")
    axis.plot(t_grid, metrics["median_t"], "--", label="median")
    axis.fill_between(t_grid, np.maximum(metrics["mean_t"] - metrics["std_t"], 0), metrics["mean_t"] + metrics["std_t"], alpha=0.18, label="mean +/- std")
    axis.axvline(t_grid[H - 1], color="black", linestyle=":", label="last exact input")
    axis.set(xlabel="time", ylabel="absolute physical L2", title=f"AWFNO-H10-K5 rollout, N={n_train}")
    axis.legend()
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f"N{n_train}_l2_per_time.png", dpi=150)
    plt.close(figure)



## Full sample-size sweep



In [ ]:
results = []
for n_train in CONFIG["RUN_SAMPLE_SIZES"]:
    if n_train not in CONFIG["SAMPLE_SIZES"]:
        raise ValueError(f"Unexpected sample size: {n_train}")
    train_history, train_future = build_windows(U_train_pool[:n_train], WINDOW_STARTS)
    print(f"\nN={n_train}: histories {train_history.shape}, futures {train_future.shape}")
    fitted = fit_or_load(n_train, train_history, train_future)
    prediction, inference_median, inference_all = timed_rollout(fitted["model"], fitted["predict"], U_test)
    metrics = rollout_metrics(prediction, U_test)
    result = {
        "n_train": int(n_train), "model": "AWFNO-H10-K5-W32-L4-M16-R3",
        "parameters": int(count_parameters(fitted["model"])), "stored_real_scalars": int(count_real_scalars(fitted["model"])),
        "best_epoch": int(fitted["best_epoch"]), "validation_window_l2": float(fitted["best_val"]),
        "full_mean": metrics["full_mean"], "full_std": metrics["full_std"], "full_median": metrics["full_median"],
        "final_mean": metrics["final_mean"], "final_std": metrics["final_std"], "final_median": metrics["final_median"],
        "train_time_sec": float(fitted["train_time_sec"]), "inference_time_median_sec": inference_median,
        "inference_time_all_sec": inference_all, "checkpoint": fitted["checkpoint"],
    }
    results.append(result)
    np.savez_compressed(RESULT_DIR / f"N{n_train}_metrics.npz", mean_t=metrics["mean_t"], median_t=metrics["median_t"], std_t=metrics["std_t"], all_full=metrics["all_full"], all_final=metrics["all_final"], inference_times=np.asarray(inference_all))
    if CONFIG["SAVE_ROLLOUTS"]:
        np.save(RESULT_DIR / f"N{n_train}_rollout.npy", prediction.astype(np.float32))
    plot_history(fitted["history"], n_train)
    plot_l2_time(metrics, n_train)
    print("\nTest result")
    print("-----------")
    print(f"model             : {result['model']}")
    print(f"N_train           : {n_train}")
    print(f"parameters        : {result['parameters']:,}")
    print(f"full mean/std     : {result['full_mean']:.6e} / {result['full_std']:.6e}")
    print(f"full median       : {result['full_median']:.6e}")
    print(f"final mean/std    : {result['final_mean']:.6e} / {result['final_std']:.6e}")
    print(f"final median      : {result['final_median']:.6e}")
    print(f"training time     : {result['train_time_sec']:.2f} s")
    print(f"inference median  : {result['inference_time_median_sec']:.4f} s")
    del train_history, train_future, fitted, prediction, metrics
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

csv_path = RESULT_DIR / "burgers_awfno_h10_k5_summary.csv"
json_path = RESULT_DIR / "burgers_awfno_h10_k5_summary.json"
pd.DataFrame(results).to_csv(csv_path, index=False)
with json_path.open("w") as handle:
    json.dump(results, handle, indent=2)
display(pd.DataFrame(results))

print("\n" + "=" * 150)
print("BURGERS 1D AWFNO-H10-K5: ABSOLUTE PHYSICAL L2")
print("=" * 150)
for row in results:
    print(f"N={row['n_train']:4d} | full mean {row['full_mean']:.6e} | std {row['full_std']:.6e} | median {row['full_median']:.6e} | final mean {row['final_mean']:.6e} | final median {row['final_median']:.6e} | params {row['parameters']:,} | train {row['train_time_sec']:.2f}s | inference {row['inference_time_median_sec']:.4f}s")
print("=" * 150)
print("Saved CSV :", csv_path)
print("Saved JSON:", json_path)



## Release the Colab runtime

Run this only after the summary paths above have been printed.



In [ ]:
if CONFIG["STOP_RUNTIME_AT_END"]:
    print("All results are saved. Releasing the Colab runtime.")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        os.kill(os.getpid(), signal.SIGTERM)
